In [1]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import cv2
import imageio

# Import your local modules
# Ensure conditional_unet_depth.py and diffusion.py are in the same folder
from conditional_unet_depth import ConditionalUNetDepth
from diffusion import GaussianDiffusion

# --- Configuration ---
CONFIG = {
    "img_size": 128,        # Resize all images to this (e.g., 128 or 256)
    "batch_size": 16,       # Adjust based on VRAM (8GB should handle 16-32 at 128x128)
    "lr": 1e-4,
    "epochs": 100,
    "timesteps": 1000,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "synthetic_root": "./dataset/synthetic_dataset", # TODO Update these paths!
    "real_val_root": "./dataset/real-val/d435_dataset"
}

print(f"Running on device: {CONFIG['device']}")


ImportError: attempted relative import with no known parent package

In [ ]:

# --- Dataset Class ---

class ClearGraspDataset(Dataset):
    def __init__(self, root_dir, split="synthetic", img_size=128, max_depth=3.0):
        """
        Args:
            root_dir: Path to dataset folder.
            split: "synthetic" (training) or "real" (validation/test).
            img_size: Target size for resizing (square).
            max_depth: Maximum depth value (in meters) for normalization.
                       ClearGrasp usually clips around 3m-10m. 3.0m is a safe focus for manipulation.
        """
        self.root_dir = root_dir
        self.split = split
        self.img_size = img_size
        self.max_depth = max_depth

        self.samples = []

        if self.split == "synthetic":
            # Synthetic structure: Class/depth-imgs-rectified/*.exr
            # We need to find matching RGB and Mask files.
            # Assumes structure: class_name/depth-imgs-rectified/000.exr
            #                    class_name/rgb-imgs/000.jpg
            #                    class_name/segmentation-masks/000.png (or similar)

            # Walk through all class folders
            for class_folder in glob.glob(os.path.join(root_dir, "*")):
                if not os.path.isdir(class_folder): continue

                depth_folder = os.path.join(class_folder, "depth-imgs-rectified")
                rgb_folder = os.path.join(class_folder, "rgb-imgs")
                # Note: Check the exact name of your mask folder in the synthetic dataset!
                # Often it is 'segmentation-masks' or similar.
                mask_folder = os.path.join(class_folder, "segmentation-masks")

                if not os.path.exists(depth_folder): continue

                depth_files = sorted(glob.glob(os.path.join(depth_folder, "*.exr")))

                for df in depth_files:
                    basename = os.path.basename(df).split("-")[0] # e.g. "000000000"

                    # Construct paths for RGB and Mask
                    # Adjust extensions/naming patterns based on your actual file listing!
                    rgb_path = os.path.join(rgb_folder, f"{basename}-rgb.jpg")
                    mask_path = os.path.join(mask_folder, f"{basename}-segmentation-mask.png")

                    if os.path.exists(rgb_path) and os.path.exists(mask_path):
                        self.samples.append({
                            "depth": df,
                            "rgb": rgb_path,
                            "mask": mask_path
                        })

        elif self.split == "real":
            # Real Val/Test structure (flat folder):
            # 000-opaque-depth-img.exr (Target)
            # 000-transparent-depth-img.exr (Input Bad Depth)
            # 000-transparent-rgb-img.jpg (Input RGB)

            # We iterate over opaque depth (ground truth targets)
            target_files = sorted(glob.glob(os.path.join(root_dir, "*-opaque-depth-img.exr")))

            for tf in target_files:
                # e.g. .../000000000-opaque-depth-img.exr
                prefix = tf.split("-opaque")[0] # .../000000000

                input_depth_path = f"{prefix}-transparent-depth-img.exr"
                input_rgb_path = f"{prefix}-transparent-rgb-img.jpg"

                if os.path.exists(input_depth_path) and os.path.exists(input_rgb_path):
                    self.samples.append({
                        "target_depth": tf,
                        "input_depth": input_depth_path,
                        "rgb": input_rgb_path
                    })

        print(f"Found {len(self.samples)} samples for split: {split}")

    def load_exr(self, path):
        # Load EXR file using OpenCV
        # flag -1 (IMREAD_UNCHANGED) is critical for 16-bit/32-bit depth
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        # Handle potential multi-channel EXR (take first channel if 3)
        if len(img.shape) == 3:
            img = img[:, :, 0]
        return img.astype(np.float32)

    def process_image(self, img, is_depth=False):
        # 1. Center Crop to square
        h, w = img.shape[:2]
        min_dim = min(h, w)
        top = (h - min_dim) // 2
        left = (w - min_dim) // 2
        img = img[top:top+min_dim, left:left+min_dim]

        # 2. Resize
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST if is_depth else cv2.INTER_LINEAR)

        # 3. To Tensor
        if not is_depth:
            # RGB: (H, W, 3) -> (3, H, W), scale 0-255 to [-1, 1]
            img = torch.from_numpy(img).permute(2, 0, 1).float()
            img = (img / 127.5) - 1.0
        else:
            # Depth: (H, W) -> (1, H, W)
            img = torch.from_numpy(img).unsqueeze(0).float()
            # Normalize depth: [0, max_depth] -> [-1, 1]
            # Clip to max depth first
            img = torch.clamp(img, 0, self.max_depth)
            # Scale to [0, 1]
            img = img / self.max_depth
            # Scale to [-1, 1] for diffusion
            img = (img * 2.0) - 1.0

        return img

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # --- Load RGB ---
        rgb = cv2.imread(sample["rgb"])
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB) # OpenCV loads BGR
        rgb_tensor = self.process_image(rgb, is_depth=False)

        if self.split == "synthetic":
            # --- Synthetic: Create Corruption ---
            target_depth_raw = self.load_exr(sample["depth"])

            # Load Mask (0=Background, 255=Object)
            # Mask filenames might need adjustment based on exact dataset names
            mask = cv2.imread(sample["mask"], cv2.IMREAD_GRAYSCALE)

            # Ensure mask and depth match size (sometimes synthetic data has mismatches)
            if mask.shape != target_depth_raw.shape:
                mask = cv2.resize(mask, (target_depth_raw.shape[1], target_depth_raw.shape[0]), interpolation=cv2.INTER_NEAREST)

            # Process Target (Perfect) Depth
            target_depth_tensor = self.process_image(target_depth_raw, is_depth=True)

            # Create Input (Corrupted) Depth
            # Important: Crop/Resize mask SAME as depth
            mask_tensor = self.process_image(mask, is_depth=True) # reusing depth logic for single channel
            # mask_tensor is [-1, 1] now due to process_image logic for depth.
            # Let's fix logic: > 0 means transparent object.
            # Since mask was 0 or 255, normalized it becomes -1 or 1 approx.

            input_depth_tensor = target_depth_tensor.clone()
            # Corruption: Set pixels where mask indicates object to "empty" (-1.0 in our normalized space)
            input_depth_tensor[mask_tensor > 0.0] = -1.0

        else:
            # --- Real: Load existing pair ---
            target_depth_raw = self.load_exr(sample["target_depth"])
            input_depth_raw = self.load_exr(sample["input_depth"])

            target_depth_tensor = self.process_image(target_depth_raw, is_depth=True)
            input_depth_tensor = self.process_image(input_depth_raw, is_depth=True)

        # Conditioning = RGB (3) + Input Bad Depth (1)
        conditioning = torch.cat([rgb_tensor, input_depth_tensor], dim=0)

        return {
            "pixel_values": target_depth_tensor, # What we want to predict (x0)
            "conditioning": conditioning         # What we know
        }

    def __len__(self):
        return len(self.samples)


In [ ]:

# --- Sanity Check Block ---
if __name__ == "__main__":
    # Create dummy datasets (assuming you have pointed paths correctly)
    # If you don't have data yet, this will just print 0 samples
    try:
        ds = ClearGraspDataset(CONFIG['real_val_root'], split="real", img_size=128)
        if len(ds) > 0:
            item = ds[0]
            print("RGBD Cond Shape:", item["conditioning"].shape)
            print("Target Depth Shape:", item["pixel_values"].shape)
            print("Depth Range:", item["pixel_values"].min().item(), item["pixel_values"].max().item())

            # Visualization
            cond_img = item["conditioning"][:3].permute(1, 2, 0).cpu().numpy() # RGB
            cond_img = (cond_img + 1) / 2.0 # to [0, 1]

            bad_depth = item["conditioning"][3].cpu().numpy()
            good_depth = item["pixel_values"][0].cpu().numpy()

            fig, ax = plt.subplots(1, 3, figsize=(12, 4))
            ax[0].imshow(cond_img)
            ax[0].set_title("Input RGB")
            ax[1].imshow(bad_depth, cmap='inferno')
            ax[1].set_title("Input Bad Depth")
            ax[2].imshow(good_depth, cmap='inferno')
            ax[2].set_title("Target Perfect Depth")
            plt.show()
    except Exception as e:
        print(f"Could not load dataset for sanity check: {e}")
        print("Please verify paths in CONFIG")